# 检索前过滤无权访问的资料

相似度高不代表用户有权查看。这个例子使用同一份南瓜书 PDF 的片段，正文问题和受限页面都从问题集重新读取；访问级别只是教学设定，不表示《南瓜书》真的包含保密内容。

下面先检查连续属性离散化的检索结果，比较过滤前后是否会带出无权访问的资料。这里设定的访问范围恰好挡住了回答所需内容，因此过滤后的正确结果是停止回答。随后再用一个普通问题确认过滤没有误删允许访问的资料，并换一道 DBSCAN 问题重复检查。

In [1]:
import sys
from pathlib import Path


def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / "data" / "dataset/manifest.json").is_file():
            return folder
    raise FileNotFoundError("没有找到教程数据目录，请从本节所在目录运行。")


course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

In [2]:
from common.eval_utils import emit_tutorial_audit

import json
from common.eval_utils import (
    build_bm25_chunk_search,
    load_query_catalog,
    load_default_collection,
    load_default_chunks,
)
from common.nontraining_utils import load_annotation, load_query_controls

cases = {item["id"]: item for item in load_query_catalog()}
case = cases["restricted_page_access"]
normal_case = cases["slater_strong_duality"]
dbscan_case = cases["dbscan_scope_filter"]
chunks = load_default_chunks(load_default_collection())
access_controls = load_query_controls(case["id"])
restricted_pages = set(access_controls["restricted_pages"])
main_useful_terms = tuple(term for term in ("连续属性", "离散化") if term in case["query"])

for chunk in chunks:
    chunk["allowed_for_reader"] = not bool(restricted_pages.intersection(chunk["pages"]))

def standard_metrics(results, expected_pages):
    pages = [int(page) for item in results for page in item.pages]
    expected = set(int(page) for page in expected_pages)
    if not expected:
        # 无答案题没有“必要页”，页面指标只保留真实的无答案语义；
        # 受限结果数量另用 comparison 保存，不能伪装成覆盖率。
        return {'pages': pages, 'first_required_rank': None, 'required_page_coverage': 0.0}
    found = {page for page in pages if page in expected}
    first = next((rank for rank, page in enumerate(pages, 1) if page in expected), None)
    return {'pages': pages, 'first_required_rank': first,
            'required_page_coverage': len(found) / len(expected)}

def emit_standard(method, role, case_id, before, after, expected_pages, restricted_pages=None, check_purpose=None, comparison=None):
    payload = {'case_id': case_id, 'method': method, 'role': role,
               'before': standard_metrics(before, expected_pages),
               'after': standard_metrics(after, expected_pages)}
    if check_purpose:
        payload['check_purpose'] = check_purpose
    if comparison:
        payload['comparison'] = comparison
    emit_tutorial_audit(payload)

print("问题：", case["query"])
print("无权访问的页码：", sorted(restricted_pages))
print("片段总数：", len(chunks))

问题： 检索与连续属性离散化有关的资料。
无权访问的页码： [50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60]
片段总数： 987


## 比较过滤前后

基础方案直接在全部片段中排序。改动后先留下普通读者可看的片段，再执行同一套 BM25 检索。不能先查全库再隐藏结果，因为标题、分数或片段内容可能已经泄露；过滤后还要分别检查正文查询是否被安全地阻止，以及一个正常查询是否仍有允许的有用资料。

In [3]:
result_k = 30
allowed_chunks = [chunk for chunk in chunks if chunk["allowed_for_reader"]]

def restricted_count(results, page_set):
    return sum(bool(page_set.intersection(item.pages)) for item in results)

def useful_matches(results, terms):
    return [item for item in results if all(term.lower() in item.text.lower() for term in terms)]

main_before = build_bm25_chunk_search(chunks)(case["query"], top_k=result_k)
main_after = build_bm25_chunk_search(allowed_chunks)(case["query"], top_k=result_k)
main_before_restricted = restricted_count(main_before, restricted_pages)
main_after_restricted = restricted_count(main_after, restricted_pages)
main_before_useful = useful_matches(main_before, main_useful_terms)
main_after_useful = useful_matches(main_after, main_useful_terms)
main_annotation = load_annotation(case['id'])

print("\n正文查询改前结果页：", [item.pages[0] for item in main_before])
print("正文查询改后结果页：", [item.pages[0] for item in main_after])
print("正文查询受限结果数量：", main_before_restricted, "→", main_after_restricted)
print("正文查询可回答候选数量（同时含“连续属性”和“离散化”）：", len(main_before_useful), "→", len(main_after_useful))
if main_after_useful:
    print("正文结论：过滤后仍有允许的可用资料，可以继续回答。")
else:
    print("正文结论：阻止泄露；过滤后没有允许的可用答案，不宣称仍能回答。")

main_evaluation = main_annotation["evaluation"]
assert main_before_restricted >= main_evaluation["restricted_results_before_min"]
assert main_after_restricted == main_evaluation["restricted_results_after"]
assert not main_after_useful
emit_standard('按资料范围筛选', 'main', case['id'], main_before, main_after, main_annotation['expected_pages'], restricted_pages, '说明不适用或限制', comparison={'name': '受限结果数量', 'before': main_before_restricted, 'after': main_after_restricted, 'higher_is_better': False})

normal_before = build_bm25_chunk_search(chunks)(normal_case["query"], top_k=result_k)
normal_after = build_bm25_chunk_search(allowed_chunks)(normal_case["query"], top_k=result_k)
normal_annotation = load_annotation(normal_case["id"])
normal_target_pages = set(normal_annotation["expected_pages"])
normal_after_useful = useful_matches(normal_after, ("Slater", "强对偶"))
normal_target_allowed = any(normal_target_pages.intersection(item.pages) for item in normal_after)
print("\n正常查询改前/改后：", normal_case["query"])
print("正常查询结果页：", [item.pages[0] for item in normal_before], "→", [item.pages[0] for item in normal_after])
print("正常查询受限结果数量：", restricted_count(normal_before, restricted_pages), "→", restricted_count(normal_after, restricted_pages))
print("正常查询过滤后允许的有用资料页：", [item.pages[0] for item in normal_after_useful])
print("正常查询仍找到允许的目标页：", normal_target_allowed)
assert restricted_count(normal_after, restricted_pages) == 0
assert normal_after_useful
assert normal_target_allowed

dbscan_controls = load_query_controls(dbscan_case["id"])
dbscan_restricted_pages = set(dbscan_controls["restricted_pages"])
dbscan_chunks = load_default_chunks(load_default_collection())
for chunk in dbscan_chunks:
    chunk["allowed_for_reader"] = not bool(dbscan_restricted_pages.intersection(chunk["pages"]))
dbscan_allowed_chunks = [chunk for chunk in dbscan_chunks if chunk["allowed_for_reader"]]
dbscan_before = build_bm25_chunk_search(dbscan_chunks)(dbscan_case["query"], top_k=5)
dbscan_after = build_bm25_chunk_search(dbscan_allowed_chunks)(dbscan_case["query"], top_k=5)
dbscan_annotation = load_annotation(dbscan_case['id'])
dbscan_before_useful = useful_matches(dbscan_before, dbscan_annotation["expected_keywords"])
dbscan_after_useful = useful_matches(dbscan_after, dbscan_annotation["expected_keywords"])
print("\n换一道题检查：", dbscan_case["query"])
print("无权访问的页码：", sorted(dbscan_restricted_pages))
print("DBSCAN 改前/改后结果页：", [item.pages[0] for item in dbscan_before], "→", [item.pages[0] for item in dbscan_after])
print("DBSCAN 受限结果数量：", restricted_count(dbscan_before, dbscan_restricted_pages), "→", restricted_count(dbscan_after, dbscan_restricted_pages))
print("DBSCAN 可回答候选数量（同时含 DBSCAN 和密度聚类）：", len(dbscan_before_useful), "→", len(dbscan_after_useful))
print("DBSCAN 结论：过滤后不返回受限范围资料，当前查询显示阻止泄露。")
dbscan_evaluation = dbscan_annotation["evaluation"]
assert restricted_count(dbscan_before, dbscan_restricted_pages) >= dbscan_evaluation["restricted_results_before_min"]
assert restricted_count(dbscan_after, dbscan_restricted_pages) == dbscan_evaluation["restricted_results_after"]
assert dbscan_before_useful
assert not dbscan_after_useful
emit_standard('按资料范围筛选', 'check', dbscan_case['id'], dbscan_before, dbscan_after, dbscan_annotation['expected_pages'], dbscan_restricted_pages, '说明不适用或限制', comparison={'name': '受限结果数量', 'before': restricted_count(dbscan_before, dbscan_restricted_pages), 'after': restricted_count(dbscan_after, dbscan_restricted_pages), 'higher_is_better': False})


正文查询改前结果页： [51, 51, 31, 52, 111, 80, 182, 142, 109, 52, 122, 138, 112, 14, 31, 139, 139, 45, 113, 161, 45, 2, 95, 160, 188, 51, 103, 35, 162, 58]
正文查询改后结果页： [31, 111, 80, 182, 142, 109, 122, 138, 112, 31, 14, 139, 45, 113, 139, 45, 161, 2, 160, 188, 95, 103, 35, 45, 78, 119, 162, 21, 111, 141]
正文查询受限结果数量： 6 → 0
正文查询可回答候选数量（同时含“连续属性”和“离散化”）： 3 → 0
正文结论：阻止泄露；过滤后没有允许的可用答案，不宣称仍能回答。


正常查询改前/改后： Slater 条件如何保证强对偶成立？
正常查询结果页： [63, 64, 64, 63, 63, 66, 64, 62, 64, 65, 5, 141, 61, 184, 155, 63, 95, 39, 184, 181, 65, 52, 152, 27, 61, 69, 180, 154, 75, 75] → [63, 64, 64, 63, 63, 66, 64, 62, 64, 65, 141, 5, 61, 184, 155, 63, 95, 184, 39, 181, 65, 152, 61, 27, 69, 154, 180, 75, 164, 75]
正常查询受限结果数量： 1 → 0
正常查询过滤后允许的有用资料页： [63, 64, 66]
正常查询仍找到允许的目标页： True



换一道题检查： 检索与 DBSCAN 密度聚类有关的资料。
无权访问的页码： [112, 113, 114, 115, 116, 117, 118, 119]
DBSCAN 改前/改后结果页： [118, 112, 118, 8, 109] → [8, 109, 8, 142, 164]
DBSCAN 受限结果数量： 3 → 0
DBSCAN 可回答候选数量（同时含 DBSCAN 和密度聚类）： 1 → 0
DBSCAN 结论：过滤后不返回受限范围资料，当前查询显示阻止泄露。



## 结论

连续属性问题在过滤前会返回无权访问的资料，过滤后这类结果降为 0，同时也失去了完整答案，因此程序应停止回答。两道资料范围边界题没有必要答案页，页面覆盖率保持为 0；审计记录另外保存受限结果数量：正文题 6→0，DBSCAN 复查题 3→0。换成 Slater 条件问题后，过滤仍能留下允许访问的正确资料。实际使用时，缓存、日志、来源标注和回答模型都要遵守相同的访问限制；本节只演示检索前的过滤。

## 本页导航

- 章节入口：[本章 README](README.md)
- 运行准备：[C7 统一运行准备](../README.md#运行准备)
- 相关下一步：[改写或拆分问题](改写检索问题.ipynb)

